# Lab 1: Tokenisation and embeddings

In this lab, you will build an understanding of how text can be transformed into representations that computers can process and learn from. Specifically, you will explore two key concepts: *tokenisation* and *embeddings*. Tokenisation splits text into smaller units such as words, subwords, or characters. Embeddings are dense, fixed-size vector representations of tokens in a continuous space.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

## Part 1: Tokenisation

In the first part of the lab, you will code and analyse a tokeniser based on the Byte Pair Encoding (BPE) algorithm.

### Utility functions

The BPE tokeniser transforms text into a list of integers representing tokens. As a warm-up, you will implement two utility functions on such lists. To simplify things, we define a shorthand for the type of pairs of integers:

In [ ]:
type Pair = tuple[int, int]

#### 🧩 Task 1.01: Counting pairs

Write a function that counts all occurrences of pairs of consecutive token IDs in a given list. The function should return a dictionary that maps each pair to its count. Skip counts that are zero.

In [ ]:
def count(ids: list[int]) -> dict[Pair, int]:
    counts  = {}
    for i in range(len(ids) -1):
        pair = (ids[i], ids[i + 1])
        counts[pair] = counts.get(pair,0)+1
    return counts

#### 🧩 Task 1.02: Replacing pairs

Write a function that traverses a list of token IDs from left to right and replaces all occurrences of a specified pair of consecutive IDs by a new ID. The function should return the modified list.

In [3]:
def replace(ids: list[int], pair: Pair, new_id: int) -> list[int]:
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

### Encoding and decoding

The next cell contains the core code for the tokeniser in the form of a class `Tokenizer`. This class implements two methods: `encode()` converts an input text to a list of token IDs by exhaustively applying rules for merging pairs of consecutive IDs (stored in the dictionary `self.merges`), and `decode()` reverses this process.

**Note that the set of merge rules is initially empty; you will add rules in Task&nbsp;1.04.**

In [ ]:
class Tokenizer:
    def __init__(self: "Tokenizer") -> None:
        # Input: Kuch nahi | Output: Empty dictionary {} jo merge rules store karegi (e.g., (97, 98): 256)
        self.merges = {}                                        # type: dict[Pair, int]

        # Input: 0 se 255 tak integers | Output: Dict {0: b'\x00', 1: b'\x01', ... 255: b'\xff'}
        self.vocab = {i: bytes([i]) for i in range(2**8)}       # type: dict[int, bytes]

    def encode(self, text: str) -> list[int]:
        # Input: Python string (e.g., "hello") | Output: Bytes ki integer list (e.g., [104, 101, 108, 108, 111])
        ids = list(text.encode("utf-8"))                         

        while True:
            # Input: Token IDs ki list | Output: Pairs aur unki frequency ka Dict (e.g., {(104, 101): 1})
            counts = count(ids)  #yeh left se right check karta hai aur har pair ki frequency count karta hai
            #and is ki output aik dictionary hoti hai jisme keys pairs hote hain aur values unki frequency hoti hai
            # example here: agar ids = [104, 101, 108, 108, 111] (jo "hello" ka byte representation hai) toh counts = {(104, 101): 1, (101, 108): 1, (108, 108): 1, (108, 111): 1}                             

            # Input: Text ke pairs aur Trained merges ki keys | Output: Dono mai common pairs ka Set (Intersection)
            mergeable_pairs = counts.keys() & self.merges.keys() 
            #example here: agar counts = {(104, 101): 1, (101, 108): 1, (108, 108): 1, (108, 111): 1} aur self.merges = {(101, 108): 256, (108, 108): 257} toh mergeable_pairs = {(101, 108), (108, 108)}
            #its difficult to explain in urdu but basically it finds the common pairs between the counts and the trained merges, but iss ka kaam hai
            # to identify which pairs in the current text can be merged based on the trained merges.
            

            # Input: mergeable_pairs Set ki length | Output: Agar length 0 hai toh loop khatam (Break)
            if len(mergeable_pairs) == 0:                        
                break   

           # Input: Common pairs Set aur `self.merges` Dict | Output: Target Pair jisko sab se pehle merge karna hai

# Example: 
# Agar mergeable_pairs = {(101, 108), (108, 108)} aur self.merges = {(101, 108): 256, (108, 108): 257}
# Toh `to_merge` = (101, 108) hoga kyun ke `self.merges.get` ke mutabiq 256 < 257 hai.

# Kyun `min()` use hota hai?
# Training ke waqt jo rules PEHLE seekhe jate hain unhe CHOTA ID milta hai (e.g., 256, 257). 
# `min()` use karne se training priority preserve rehti hai.

# What if we use `max()`?
# Agar hum `max()` use karenge toh bad mai seekhe gaye (bade ID wale) rules pehle apply ho jayenge.
# For example: Agar rules hain {(A, B): 256, (256, C): 257}:
# - `min()` pehle (A, B) ko merge karke 256 banayega, taake agle step mai (256, C) merge ho sake.
# - `max()` pehle (256, C) dhoondne ki koshish karega, jo abhi list mai exist hi nahi karta kyun ke (A, B) abhi tak merge hokar 256 bana hi nahi! Is waja se tokenisation break ho jayegi.
            to_merge = min(mergeable_pairs, key=self.merges.get)
            # Input: Current IDs list, Target Pair, New ID | Output: Purane pair ki jagah naya ID dali hui updated List
            ids = replace(ids, to_merge, self.merges[to_merge])   

        # Input: Processed IDs ki final list | Output: Token IDs ki list return hoti hai
        return ids

    def decode(self, ids: list[int]) -> str:
        # Input: Token IDs ki list | Output: Har ID ke bytes look up kar ke ek single Bytes object banata hai
        # Phir `.decode("utf-8")` un bytes ko waapis normal readable String mai convert kar ke return karta hai
        #example : agar ids = [104, 101, 108, 108, 111] toh self.vocab[i] for i in ids will give [b'h', b'e', b'l', b'l', b'o']
        #and then b''.join(...) will give b'hello' and finally .decode("utf-8") will give 'hello'
        return b"".join((self.vocab[i] for i in ids)).decode("utf-8")

#### 🎓 Task 1.03: Encoding and decoding

Explain how the code implements the BPE algorithm. Use the following steps to check your understanding:

**Step&nbsp;1.** Annotate the attributes and methods of the `Tokenizer` class with their Python types. In particular, what is the type of `self.merges`? Use the `Pair` shorthand. What does a merge rule look like? <br>


1- The type of the 'self.merges' is dictionary dict [Pair,int]<br>
2- A merge rule could look look Pair[int,int] --> int which maps two consecutive IDS into their corresponding merge ID. 

**Step&nbsp;2.** Explain how the implementation chooses which merge rule to apply. Provide an example that illustrates the logic. Construct the example such that you get a different result when you use `max()` instead of `min()`.<br>


Given a text the encode method does the following:
<ol>

<li> ids = Create a list of byte values using utf-8 </li>
<li>Continue untill no merges can be made</li>
<li>counts = map each pair to it's frequency</li>
<li>mergeble pairs = intersection of counts and self.merges, finds the pairs that have merge rules </li>
<li>if this set union is empty their no pair to merge --> done!</li>
<li> amongst the mergeble pairs find the one with smallest id, preserves the order of learnt merge rules during training</li> 
<li> replace the occurance of the selected pair
</ol>

using max instead of min would reverse the order of the merge ruels meaning instead of applying the first learnt merge rule we apply the most recent one, this will not preserve the the same tokanization bahavior. Because earlier merges combine the most common pairs while later merges combine more rarer pairs. Tokanizat8ion is also order dependent.

Example:<br>
with min standard <br>
iteration 1:<br>
text = "aaab"<br>
ids = [97,97,97,98]<br>
counts = {(97,97): 2,(97,98):1}<br>
merges = {(97,97)-> 256, (256,97) --> 257,(257,98) --> 258}<br>
mergeble_pairs = [(97,97)]<br>
to_merge = (97,97)<br>
id = [256,97,98]<br>

iteration 2:<br>
ids = [256,97,98]<br>
counts = {(256,97): 1,(97,98):1}<br>
merges = {(97,97)-> 256, (256,97) --> 257,(257,98) --> 258}<br>
mergeble_pairs = [(256,97)]<br>
to_merge = (256,97)<br>
id = [257,98]<br>

iteration 3:<br>
ids = [257,98]<br>
counts = {(257,98):1}<br>
merges = {(97,97)-> 256, (256,97) --> 257,(257,98) --> 258}<br>
mergeble_pairs = [(257,98)]<br>
to_merge = (257,98)<br>
id = [258]<br>

with max standard we try to apply 258 first which does not exist so it'd fail. In general using max results in suboptimal and potentially inconsisten tokanization.


### Training a tokeniser

Upon initialisation, a tokeniser has an empty set of merge rules. Your next task is to complete the BPE algorithm and write code to learn these merge rules from a text.

#### 🎓 Task 1.04: Training a tokeniser

Write a function that induces a BPE tokeniser from a given text. The function should take the text (a string) and a target vocabulary size as input and return the trained tokeniser.

In [ ]:
def from_text(text: str, vocab_size: int) -> Tokenizer:
    # Naya tokenizer banao -- isme merges aur vocab dono honge
    tok = Tokenizer()
    
    # Text ko UTF-8 bytes mein convert karo -- har character ka numerical value
    # Example: "hello" = [104, 101, 108, 108, 111]
    ids = list(text.encode("utf-8"))
    
    # Index 256 se shuru karo kyunke 0-255 tak base bytes hain
    # Har naya merge rule ko ek naya ID doge, 256 se shuru karke
    index = 256
    
    # Jab tak vocab size na poucch jaye, merge rules seekho
    while index < vocab_size:
        # Progress bar print karo -- kitne merges hogaye
        print(f"Merge {index-256}/{vocab_size-256}", end='\r')
        
        # Sare consecutive pairs count karo -- konsa pair kitni baar aya
        # Example: agar "ll" 50 baar aya tou {(108, 108): 50}
        counts = count(ids)
        
        # Agar koi pair hi nahi raha (sab merge hogaye) tou ruko
        if len(counts) == 0:
            break
        
        # Sabse zyada frequency nikal lo -- yeh sabse common pair hoga
        # Example: max_count = 50 matlab (108, 108) 50 bar aya
        max_count = max(counts.values())
        
        # Jo pair sabse zyada aya usey select karo
        # Agar do pairs barabar frequency mein hain tou lexicographically smallest (numerically smallest) lo
        # Yeh deterministic rakhne ke liye hai -- same result har bar
        to_merge = min(p for p in counts if counts[p] == max_count)
        
        # Merge rule ko save karo: "yeh pair (to_merge) ab index ke naam se ban gaya"
        # Example: tok.merges[(108, 108)] = 256 -- matlab "ll" ab token 256 hai
        tok.merges[to_merge] = index
        
        # Naye token ko vocab mein add karo
        # Do bytes ko concatenate karo (jaise 'l' + 'l' = 'll')
        # Example: tok.vocab[256] = b'll'
        tok.vocab[index] = tok.vocab[to_merge[0]] + tok.vocab[to_merge[1]]
        
        # Ab sare occurrences of to_merge ko naye index se replace karo
        # Example: [104, 108, 108, 111] mein se (108, 108) ko 256 se badlo
        # Result: [104, 256, 111]
        ids = replace(ids, to_merge, index)
        
        # Agle iteration ke liye next merge ID ready karo
        index += 1
    
    # Trained tokenizer return karo -- isme sare merge rules hain
    return tok


# ============================================================
# YAAD RAKNE WALI BAATEIN:
# ============================================================
# 1. Greedy BPE: har iteration mein sabse frequent pair merge karo
# 2. Index 256+ se naaye tokens banate hain (0-255 base bytes)
# 3. max_count --> sabse frequent pair nikal lo
# 4. Tie-break: lexicographically smallest (smallest pair numerically)
# 5. Har merge ke baad ids update hote hain -- agle iteration mein ab woh naya token hai
# 6. Vocab bhi update hota hai -- decode ke liye bytes sequence rakhte hain

To help you test your implementation, we provide three text files together with tokenisers trained on these files. Each text file contains the first 1&nbsp;million Unicode characters in a language-specific Wikipedia:

| Text file | Tokeniser file | Wikipedia |
|---|---|---|
| `wiki-en-1m.txt` | `wiki-en-1m.tok` | [Simple English](https://simple.wikipedia.org/) |
| `wiki-is-1m.txt` | `wiki-is-1m.tok` | [Icelandic](https://is.wikipedia.org/) |
| `wiki-sv-1m.txt` | `wiki-sv-1m.tok` | [Swedish](https://sv.wikipedia.org/) |

A tokeniser file consists of lines specifying merge rules. For example, the first line in the tokeniser file for Swedish is `101 114`, which expresses that this rule combines the token with ID 101 (`e`) and the token with ID 114 (`r`). The ID of the new token (`er`) is 256 plus the (zero-indexed) line number on which the rule is found. The following code saves a `Tokenizer` to a file with this format:

In [ ]:
def save(tokenizer: Tokenizer, filename: str) -> None:
    with open(filename, "w") as f:
        for fst, snd in tokenizer.merges:
            print(f"{fst} {snd}", file=f)

with open("wiki-sv-1m.txt", "r",encoding="utf-8") as f:
    text = f.read()

tokenizer = from_text(text, vocab_size=1024)
save(tokenizer,"tokenizer.txt")
print("Tokenizer saved to tokenizer.txt")


Initial ids length: 1001360
Tokenizer saved to tokenizer.txt


#### 🎓 Task 1.05: Tokenisation quirks

Prompt [ChatGPT](https://chatgpt.com/) to reverse the letters in the following words:

```
creativecommons
MERCHANTABILITY
NSNotification
authentication
```

How many of these words come out right? What happens when you modify the prompt and explicitly disable “thinking” and external tools? What could be the problem when words come out wrong? Generate ideas by inspecting the words in Tiktokenizer. Try to come up with other prompts that illustrate problems related to tokenisation. 


<br>
1- Only authentication come out right "authentication"<br>


2- When thinking and external tools disabled the result was the same. The isssue is not with chatgps thinking but the tokanization process. Tokanization defines what the model "sees" as the atomic unit not individual characters.<br>


3- Words that have repeate letter, mixed casing or prefix that look similar when flipped are problematic. <br>
Language models like chatgpt make a prediction as to what look right instead of doing the reversing process mechanically and it is optimized for languages not word processing. Upon prompting chatgpt with the words and their reversed version it would do the following:

1. tokanize the input words
2. reverse the tokens (not the character)
3. detokanize back to text.

this would work for single token words but not multi token words like creativecommon.

To test your code, compare your saved tokeniser to the provided tokeniser using the `diff` command line tool.

**Note that training a tokeniser can take a few minutes.**

### Tokenisation quirks

The tokeniser is a key component of language models, as it defines the minimal chunks of text the model can “see” and work with. As you will see in this section, tokenisation is also responsible for several deficiencies and unexpected behaviours of language models.

One helpful tool for experimenting with tokenisers in language models is the web app [Tiktokenizer](https://tiktokenizer.vercel.app/). This app lets you play around with, among others, [`o200k_base`](https://tiktokenizer.vercel.app/?model=o200k_base), the tokeniser used in ChatGPT&nbsp;5.2. You can also use OpenAI’s own [Tokenizer](https://platform.openai.com/tokenizer) page.

### Tokenisation and multi-linguality

Many NLP systems and the tokenisers used with them are primarily trained on English data. In the next task, you will reflect on the effect this has when they are used to process non-English data.

The *context length* of a language model is the maximum number of preceding tokens the model can condition on when predicting the next token. This number is fixed and cannot be changed after training the model. For example, the context length of GPT-2 ([Radford et al., 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)) is 1,024. 

While the context length of a language model is fixed, the amount of information that can be squeezed into this context length will depend on the tokeniser. Informally speaking, a model that needs more tokens to represent a given text cannot extract as much information from that text as one that needs fewer tokens.

#### 🎓 Task 1.06: Tokenisation and multi-linguality

Train a tokeniser on the English text file from Task&nbsp;1.04 (or use the provided one) and apply it to the same text. How many tokens does it split the text into? Based on this, what is the expected number of Unicode characters of English text that can be fit into the GPT-2 context window?

What do the numbers look like if you apply the English tokeniser to the Icelandic text instead? How do you explain the differences?

Interpreting the expected number of Unicode characters as a measure of representation efficiency, what do your results tell you about the efficiency of a language model primarily trained on English data when it is used to process non-English data? Relate these findings to the issue of tokeniser fairness.

In [31]:
with open("wiki-en-1m.txt","r",encoding="utf-8") as f:
    text_english = f.read()

tokenizer = from_text(text_english, vocab_size=1024)
tokenized_english_text = tokenizer.encode(text_english)
english_compression_ratio = len(text_english) / len(tokenized_english_text)

print("Number of tokens in English tokenizer:", len(tokenized_english_text))
print("Number of unicode characters in English text that fit in GPT-2 context window:",1024*english_compression_ratio)
print("The compression ratio is:", english_compression_ratio)


Number of tokens in English tokenizer: 379780
Number of unicode characters in English text that fit in GPT-2 context window: 2696.2978566538523
The compression ratio is: 2.6331033756385276


The tokanizer split the text into 379780 tokens and based on this all of the unicode characters of the eng

In [32]:
with open("wiki-is-1m.txt","r",encoding="utf-8") as f:
    text_icelandic = f.read()
tokenized_iclandic_text = tokenizer.encode(text_icelandic)
cr_islandic = len(text_icelandic) / len(tokenized_iclandic_text)

print("Number of tokens in English tokenizer:", len(tokenized_iclandic_text))
print("Number of unicode characters in Icelandic text that fit in GPT-2 context window:", 1024*cr_islandic)
print("The compression ratio is:", cr_islandic)


Number of tokens in English tokenizer: 754584
Number of unicode characters in Icelandic text that fit in GPT-2 context window: 1357.0391102912333
The compression ratio is: 1.3252335061437825


The difference is due to tokenization premiums, BPE tokenization length is not the same for all languages.The unfairness happens because of variable size encoding, for instance in utf-8 all english character are represented with 1 byte while for languages like Shan that have higher codepoints a character is represented with 3 bytes. 




Another issue with BPE is that it penalizes the low frequency languages. If we look att the compression rates this becomes obivous. In the above example we see that when the tokenizer trained on the english text is used to encode islandic text is results in lower compression rate. This leads to a bais towrds the language that the model was trained on thus making unfair to other languages.

## Part 2: Embeddings

In the second part of the lab, you will explore embeddings. An embedding layer is a network component that assigns each item in a finite set of elements (often called a *vocabulary*) a fixed-size vector. At first, these vectors are filled with random values, but during training, they are adjusted to suit the task at hand.

### Bag-of-words classifier

To help you build an intuition for embeddings and the vector representations learned by them, we will use a simple bag-of-words text classifier. The core part of this classifier only takes a few lines of code:

In [8]:
import torch.nn as nn


class Classifier(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.linear = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        return self.linear(self.embedding(x).mean(dim=-2))

#### 🧩 Task 1.07: Bag-of-words classifier

Explain how the bag-of-words classifier works. How does the code match the diagram you saw in the lectures? Why is there only one `nn.Embedding`, while the diagram shows three embedding layers? What does the keyword argument `dim=-2` do?

The Bag-of-words classifier works in as the following steps:

1. Build a vocabulary   = All the unique word in the training data, this creates a fixed set of tokens the model can recognize.
2. Vectorize            = for each document create a vector representing the postion/frequency 
3. Classify             = forward these vectors to a ML classifier 

The diagram in the lectures depicts a neural network pipeline as the following:<br>
Input = token IDs fed as x<br>
Embedding layer = nn.Embedding map token IDs to dense vectors<br>
Aggregation = .mean(dim = 2) pools the embedding into a signle vector per document<br>
Linear layer = nn.Linear produces the logits (unnormalized probabilities)<br>
Ouput  = probabilities for each class.<br>
The Classifier class implements a simple version of this focusing on Bow.<br>

There is one embedding layer as opposse to 3 in the diagram because all tokens share the same vocabulary and embedding space, a single embedding layer sufficeintly looks up vectors for all tokens in the batch. In the diagram the layers illustrate per-token processing before aggregation in the Classifier one layer handles this uniformly for any number of tokens.


dim = 2 specifies the dimension to average over.(batch_size,seq_len,embedding_dim) --> dim = 2 averages over seq_len, this collapses variable size sequences into fixed sized vector.



### Dataset

You will apply the classifier to a small dataset with Amazon customer reviews. This dataset is taken from [a much larger dataset](https://www.cs.jhu.edu/~mdredze/datasets/sentiment/) first described by [Blitzer et al. (2007)](https://aclanthology.org/P07-1056/).

The dataset contains whitespace-tokenised product reviews from two categories: cameras (`camera`) and music (`music`). Each review is additionally annotated for sentiment towards the product at hand: negative (`neg`) or positive (`pos`). The category and sentiment labels are prepended to the review. As an example, here is the first review from the training data:

```
music neg oh man , this sucks really bad . good thing nu-metal is dead . thrash metal is real metal , this is for posers
```

The next cell contains a custom [`Dataset`](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html) class for the review dataset. To initialise an instance of this class, you specify the name of the file containing the reviews you want to load (`filename`) and which of the two labels you want to use (`label`): product category (0) or sentiment (1).

In [3]:
from torch.utils.data import Dataset


class ReviewDataset(Dataset):
    def __init__(self, filename: str, label: int = 0) -> None:
        with open(filename) as f:
            tokenized_lines = [line.split() for line in f]
        self.items = [(tokens[2:], tokens[label]) for tokens in tokenized_lines]

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> tuple[list[str], str]:
        return self.items[idx]

### Vectoriser

To feed a review into the bag-of-words classifier, you first need to turn it into a vector of token IDs. Likewise, you need to convert the label (product category or sentiment) into an integer. The next cell contains a partially completed `ReviewVectoriser` class that handles this transformation.

In [11]:
from collections import Counter

import torch

# Type abbreviation for review–label pairs
type Item = tuple[list[str], str]   


class ReviewVectorizer:
    PAD = "[PAD]"
    UNK = "[UNK]"

    def __init__(self, dataset: ReviewDataset, n_vocab: int = 1024) -> None:
        # Unzip the dataset into reviews and labels
        # reviews : a list of the reviews 
        # labes a list of either the categories or sentiments
        reviews, labels = zip(*dataset)

        # Count the tokens and get the most common ones
        # iterate through each review and count the tokens, creates a frequency map
        counter = Counter(t for r in reviews for t in r)
        
        # collect the most common N words and reverse spot for <UNK> and <PAD> 
        most_common = [t for t, _ in counter.most_common(n_vocab - 2)]

        # Create the token-to-index and label-to-index mappings
        self.t2i = {t: i for i, t in enumerate([self.PAD, self.UNK] + most_common)}
        self.l2i = {l: i for i, l in enumerate(sorted(set(labels)))}

    def __call__(self, items: list[Item]) -> tuple[torch.Tensor, torch.Tensor]:
        reviews, labels = zip(*items)
        
        label_ids = [self.l2i[l] for l in labels]
        
        max_len = max(len(r) for r in reviews)

        review_ids = []

        for review in reviews:
            indices = [self.t2i.get(token,self.t2i[self.UNK]) for token in review]
            padding_size = max_len - len(indices)
            indices += [self.t2i[self.PAD]] * padding_size
            review_ids.append(indices)

        return torch.tensor(review_ids, dtype=torch.long), torch.tensor(label_ids, dtype=torch.long)

A `ReviewVectoriser` maps tokens and labels to IDs using two Python dictionaries. These dictionaries are set up when the vectoriser is initialised and queried when the vectoriser is called on a batch of review–label pairs. They include IDs for two special tokens:

`[PAD]` (Padding): Reviews can have different lengths, but PyTorch requires all vectors in a batch to be the same size. To handle this, the vectoriser adds `[PAD]` tokens to the end of shorter reviews so they match the length of the longest review in the batch.

`[UNK]` (Unknown): If a review contains a token that is not in the token-to-ID dictionary, the vectoriser assigns it the ID of the `[UNK]` token instead of a regular ID.

#### 🎓 Task 1.08: Vectoriser

Explain and complete the code of the vectoriser. Follow these steps:

**Step&nbsp;1.** Explain how unzipping works. What are the types of `reviews` and `labels`?<br>
Unziping produces two tuples. reviews a tuple of all the reviews and labels is either a tuple of product categories or sentimens depending on what label is feed into ReviewDataset default label = 0 labels becomes a tuple of prodcut categories.

**Step&nbsp;2.** Explain how the token-to-ID and label-to-ID mappings are constructed. How does the `most_common()` method deal with elements that occur equally often?<br>
"most_Common()" when two word has the same frequency this method returns the first one that was encounterd.<br>
in constructing the t2i = {t:i for i,t in enumerate(["PAD,"UNk"] + most_common)} flips the list into dicitonary for lookup {0:"PAD",1:"UNK",....}<br>
l2i = "sorted(set(label))" strips the duplicates so we only have unique labels.sorting ensures the "neg" always 0 and "pos" always 1.<br>

**Step&nbsp;3.** Complete the implementation of the `__call__()` method. This method should convert a list of $m$ review–label pairs into a pair $(X, y)$ where $X$ is a matrix containing the vectors with token IDs for the reviews, and $y$ is a vector containing the IDs of the corresponding labels.

### Training the classifier

With the vectoriser completed, you are ready to train a classifier. More specifically, you can train two separate classifiers: one to predict the product category of a review, and one to predict the sentiment. The next cell contains a simple training loop that you can adapt for this purpose.

In [12]:
import torch.nn.functional as F


def train(label:int = 0,n_vocab:int = 1024,num_embedding:int = 1024,embedding_dim:int=64,lr:float=0.001,nr_epochs:int=10,batch_size:int=16):
    dataset = ReviewDataset("reviews-train.txt", label)
    processor = ReviewVectorizer(dataset, n_vocab)
    model = Classifier(num_embedding, embedding_dim, len(processor.l2i))
    optimizer = torch.optim.Adam(model.parameters(), lr)
    data_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size,
        shuffle=True,
        collate_fn=processor,
    )
    for epoch in range(nr_epochs):
        model.train()
        running_loss = 0
        for bx, by in data_loader:
            optimizer.zero_grad()
            output = model(bx)
            loss = F.cross_entropy(output, by)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch}, loss: {running_loss / len(data_loader):.4f}")
    return processor, model

#### 🎓 Task 1.09: Training loop

Explain the training loop. Follow these steps:

**Step&nbsp;1.** Go through the training loop line-by-line and add comments where you find it suitable. Your comments should be detailed enough for you to explain the main steps of the loop.

**Step&nbsp;2.** The training loop contains various hard-coded values like filename, learning rate, batch size, and epoch count. This makes the code less flexible. Revise the code so that you can specify these values using keyword arguments. Use the concrete values from the code as defaults.

#### 🧩 Task 1.10: Training the classifier

Adapt the next cell to train the classifier for the two prediction tasks. Based on the loss values, which task appears to be the harder one? What is the purpose of setting a seed?<br>
We see that the loss values for the category prediction decrease more reapedly as compared to the sentiment so categorization is easier, the seed is need be need to use the same random number generator for the both.

In [18]:
torch.manual_seed(42)

# label = 0 --> category
vectorizer0, model0 = train()
print("Training the model for sentiment")
#label = 1 --> sentiment
vectorizer1, model1 = train(label=1)

Epoch 0, loss: 0.6838
Epoch 1, loss: 0.6486
Epoch 2, loss: 0.6046
Epoch 3, loss: 0.5415
Epoch 4, loss: 0.4708
Epoch 5, loss: 0.4008
Epoch 6, loss: 0.3403
Epoch 7, loss: 0.2900
Epoch 8, loss: 0.2518
Epoch 9, loss: 0.2230
Training the model for sentiment
Epoch 0, loss: 0.6922
Epoch 1, loss: 0.6823
Epoch 2, loss: 0.6774
Epoch 3, loss: 0.6673
Epoch 4, loss: 0.6545
Epoch 5, loss: 0.6381
Epoch 6, loss: 0.6183
Epoch 7, loss: 0.5975
Epoch 8, loss: 0.5747
Epoch 9, loss: 0.5508


### Inspecting the embeddings

Now that you have trained the classifier on two separate prediction tasks, it is interesting to inspect and compare the embedding vectors it learned in the process. For this you will use an online tool called the [Embedding Projector](http://projector.tensorflow.org). The next cell contains code to save the embeddings from a trained classifier in a format that can be loaded into this tool.

In [19]:
def save_embeddings(
    vectorizer: ReviewVectorizer,
    model: Classifier,
    vectors_filename: str,
    metadata_filename: str,
):
    i2t = {i: t for t, i in vectorizer.t2i.items()}
    embeddings = model.embedding.weight.detach().numpy()
    items = [(i2t[i], e) for i, e in enumerate(embeddings)]
    with open(vectors_filename, "wt") as f1, open(metadata_filename, "wt") as f2:
        for w, e in items:
            print("\t".join("{:.5f}".format(x) for x in e), file=f1)
            print(w, file=f2)

Call this code as follows:

In [ ]:
save_embeddings(vectorizer0, model0, "vectors0.tsv", "metadata0.tsv")
save_embeddings(vectorizer1, model1, "vectors1.tsv", "metadata1.tsv")

#### 🎓 Task 1.11: Inspecting the embeddings

Load the embeddings from the two classification tasks (product category classification and sentiment classification) into the Embedding Projector web app and inspect the vector spaces. How do they compare visually? Does the visualisation make sense to you?<br>
The embedding spaces for models are different for instance if inspect the word "repair" in both embedding spaces we see that the closest word to "repair" is different. The visulizaton is vector space demonstraing the embedding learnt during training. This due to the priorites of the model for instsance in sentiment analysis the model only cares about if the word is possitive or negative for instance to the model the words "delicious" and "durable" are functionally the same thus the math pulls them togather. On the other hand in categorization the model cares about what is being proccess and the "delicious" belongs to food category and "durable" to the tools and thus the math pulls them apart to avoid misscalssification.

Even if the words are the same their geometry changes depending on the what the priorities of the model during training was.


The Embedding Projector offers visualisations based on three dimensionality reduction methods: [UMAP](https://umap-learn.readthedocs.io/en/latest/), [T-SNE](https://en.m.wikipedia.org/wiki/T-distributed_stochastic_neighbor_embedding), and [PCA](https://en.m.wikipedia.org/wiki/Principal_component_analysis). Which of these seems most useful to you?<br>
All od the above mentioned methods are uaed to sutdy high dimensional data but the usefulness depends on the task at hand. Depending on what we are trying to accomplish, identify local cluster, spot gobal trends or computational speed we need to chose accordyingly amongst these methods. Since here we only care about local similarities tsne fits the task at hand better the compared to the other two methods.


Focus on the embeddings for the words *repair* and *sturdy*. Are they close to each other or far away from another? What happens if you switch to the other task? How do you explain that?<br>
In sentiment analysis this words are functionally the different to the model since it the model cares about the if a word is possitive or negative for instance the model sees sturdy as possitive and repair as negative and thus the embedding are far apart from echa other. But on the other hande if the taks at hand is categorization the model palace these words into the same category. Words are the same in both embedding space but their geometry changes depending on the priorities of the model.

### Initialisation of embedding layers

The error surfaces explored when training neural networks can be very complex. Because of this, it is crucial to choose “good” initial values for the parameters. In the final task of this lab, you will run a small experiment to see how alternative initialisations can affect a model’s performance.

In PyTorch, the weights of the embedding layer are initially set by sampling from the standard normal distribution, $\mathcal{N}(0, 1)$. However, research suggests other approaches may work better. For example, given that embedding layers share similarities with linear layers, it makes sense to use the same initialisation method for both. The default initialisation method for linear layers in PyTorch is the so-called Kaiming initialisation, introduced by [He et al. (2015)](https://www.cv-foundation.org/openaccess/content_iccv_2015/papers/He_Delving_Deep_into_ICCV_2015_paper.pdf).

#### 🧩 Task 1.12: Initialisation of embedding layers

Check the [source code of `nn.Linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear) to see how PyTorch initialises the weights of linear layers using the Kaiming initialisation method. Apply the same method to the embedding layer of your classifier and see how this affects the loss of your model and the vector spaces.

In [23]:
import torch.nn as nn 
import torch.nn.init as init 

class Classifier(nn.Module):
    def __init__(self,num_embeddings,embedding_dim,num_classes):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings,embedding_dim)
        self.linear = nn.Linear(embedding_dim,num_classes)
        
        # initilize the weigh according to a normal distribution
        init.normal(self.linear.weight,mean=0.0,std=0.01)
        # set the baises to zero 
        init.constant(self.linear.bias,0)

    def forward(self,x):
        return self.linear(self.embedding(x).mean(dim=2))
    



In [ ]:
class Review:

    def __init__(self):
        

**🥳 Congratulations on finishing lab&nbsp;1!**